Matricea de confuzie pentru 3D CNN ul antrenat

In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "Train.csv").exists() and (candidate / "Validation.csv").exists():
            return candidate
    raise FileNotFoundError("Nu gasesc radacina proiectului cu Train.csv si Validation.csv.")


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CFG, load_csv
from src.datasets import build_dataset

RUN_DIR = PROJECT_ROOT / "runs" / "cnn_3d"
BEST_MODEL_PATH = RUN_DIR / "best_model_3d_cnn.keras"
LAST_MODEL_PATH = RUN_DIR / "last_model_3d_cnn.keras"
CLASS_NAMES_PATH = RUN_DIR / "class_names.json"

MODEL_PATH = BEST_MODEL_PATH if BEST_MODEL_PATH.exists() else LAST_MODEL_PATH
missing = [p for p in [MODEL_PATH, CLASS_NAMES_PATH] if not p.exists()]
if missing:
    raise FileNotFoundError("Lipsesc fisiere dupa antrenare: " + ", ".join(str(p) for p in missing))

with CLASS_NAMES_PATH.open("r", encoding="utf-8") as f:
    class_names = json.load(f)

label_to_id = {label: idx for idx, label in enumerate(class_names)}

val_df = load_csv(CFG.CSV_VAL, require_labels=True)
val_df = val_df[val_df[CFG.COL_LABEL].isin(label_to_id)].copy()
val_df[CFG.COL_LABEL_ID] = val_df[CFG.COL_LABEL].map(label_to_id).astype("int32")
val_df = val_df.sort_values(CFG.COL_VIDEO_ID).reset_index(drop=True)

if val_df.empty:
    raise ValueError("Validation.csv nu contine exemple pentru clasele salvate in class_names.json.")

val_ds = build_dataset(
    val_df,
    split="Validation",
    training=False,
    batch_size=CFG.BATCH_SIZE,
    debug=True,
)

model = tf.keras.models.load_model(MODEL_PATH, compile=False)

all_true = []
all_pred = []
for clips, labels in val_ds:
    probs = model(clips, training=False).numpy()
    all_true.append(labels.numpy())
    all_pred.append(np.argmax(probs, axis=1).astype("int32"))

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)
cm = tf.math.confusion_matrix(
    all_true,
    all_pred,
    num_classes=len(class_names),
).numpy()

accuracy = float(np.mean(all_true == all_pred))
print(f"Model folosit: {MODEL_PATH}")
print(f"Exemple validation: {len(all_true)}")
print(f"Accuracy validation: {accuracy:.4f}")

cm_normalized = cm.astype("float32") / np.maximum(cm.sum(axis=1, keepdims=True), 1)

def plot_confusion_matrix(matrix, title, fmt, cmap="Blues"):
    fig_size = max(9, min(20, len(class_names) * 0.6))
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    image = ax.imshow(matrix, cmap=cmap)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

    ax.set_title(title)
    ax.set_xlabel("Predictie")
    ax.set_ylabel("Eticheta reala")
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=90)
    ax.set_yticklabels(class_names)

    threshold = matrix.max() / 2 if matrix.size and matrix.max() else 0
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            if value > 0:
                ax.text(
                    j,
                    i,
                    format(value, fmt),
                    ha="center",
                    va="center",
                    color="white" if value > threshold else "black",
                    fontsize=7,
                )

    plt.tight_layout()
    plt.show()


plot_confusion_matrix(cm, "Matrice de confuzie 3D CNN", "d")
plot_confusion_matrix(cm_normalized, "Matrice de confuzie 3D CNN normalizata", ".2f", cmap="Greens")

cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
cm_df


Grafic evolutie accuracy